# LeetCode 286 — Walls and Gates bằng Python

Link đề: https://leetcode.com/problems/walls-and-gates/ · Độ khó: **Medium** (Premium) · Chủ đề: Matrix, BFS (đa nguồn), Graph

Skeleton LeetCode:

```python
class Solution:
    def wallsAndGates(self, rooms: List[List[int]]) -> None:
        """
        Do not return anything, modify rooms in-place instead.
        """
```

Ghi chú: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (đường đi ngắn nhất giữa mọi cặp ô bằng Floyd–Warshall trên lưới nhỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới `m × n` gồm **tường** (`-1`), **cổng** (`0`) và **phòng trống** (`INF = 2147483647`); điền vào mỗi phòng trống **khoảng cách (số bước) tới cổng gần nhất**; không tới được thì giữ `INF`. Di chuyển **4 hướng** (lên, xuống, trái, phải), **không** qua tường.
2. Nhận ra đây là bài **đường đi ngắn nhất trên đồ thị không trọng số** ⇒ dùng **BFS**, và đặc biệt là **BFS đa nguồn (multi-source BFS)** xuất phát **đồng thời từ mọi cổng**.
3. Đi từ cách **BFS từ từng phòng trống** `O((mn)²)` → **BFS đa nguồn từ các cổng** `O(mn)` → **DFS từ từng cổng có cắt tỉa** → **BFS theo từng lớp (có đếm khoảng cách)** → **lặp nới lỏng (Bellman–Ford)**.
4. Hiểu vì sao BFS **từ cổng** tốt hơn BFS **từ phòng** (một lần quét thay cho `mn` lần quét) và vì sao **lần đầu một ô được gặp cũng là khoảng cách ngắn nhất**.
5. Tránh các lỗi kinh điển: BFS từ từng phòng, **không đánh dấu đã thăm** (lặp vô hạn / chậm), quên **cắt tỉa** trong DFS, đi **chéo**, **sửa/vượt tường**, và **gán lại tham số** thay vì sửa **tại chỗ**.

## 2. Đề bài & ví dụ

Cho lưới `rooms` kích thước `m × n` có ba loại giá trị:

- `-1`: **tường** hoặc chướng ngại vật;
- `0`: **cổng**;
- `INF`: **phòng trống**, dùng `2^31 - 1 = 2147483647` để biểu diễn vô cực (có thể giả định khoảng cách tới cổng luôn nhỏ hơn giá trị này).

Hãy điền vào mỗi **phòng trống** khoảng cách tới **cổng gần nhất**. Nếu **không thể** tới cổng nào thì giữ nguyên `INF`. (Sửa **tại chỗ**, không trả về gì.)

**Ví dụ 1**

- Input:
```
[[INF, -1,  0, INF],
 [INF, INF, INF, -1],
 [INF, -1, INF, -1],
 [  0, -1, INF, INF]]
```
- Output:
```
[[3, -1, 0, 1],
 [2,  2, 1, -1],
 [1, -1, 2, -1],
 [0, -1, 3,  4]]
```

**Ví dụ 2**

- Input: `rooms = [[-1]]`
- Output: `[[-1]]`

**Ràng buộc:** `m == rooms.length`; `n == rooms[i].length`; `1 <= m, n <= 250`; `rooms[i][j]` là `-1`, `0` hoặc `2^31 - 1`.

In [1]:
from typing import List
from collections import deque
import copy
import random
import time
import sys

INF = 2147483647
examples = [
    ([[INF, -1, 0, INF], [INF, INF, INF, -1], [INF, -1, INF, -1], [0, -1, INF, INF]],
     [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]),
    ([[-1]], [[-1]]),
]


def show(grid):
    return "\n".join("  " + " ".join(("INF" if v == INF else str(v)).rjust(3) for v in row) for row in grid)


print("ví dụ 1 (đầu vào):")
print(show(examples[0][0]))
print("kỳ vọng:")
print(show(examples[0][1]))

ví dụ 1 (đầu vào):
  INF  -1   0 INF
  INF INF INF  -1
  INF  -1 INF  -1
    0  -1 INF INF
kỳ vọng:
    3  -1   0   1
    2   2   1  -1
    1  -1   2  -1
    0  -1   3   4


## 3. Kiến thức nền

### 3.1. Lưới như một đồ thị và 4 hướng di chuyển

Mỗi ô `(r, c)` là một **nút**; hai ô kề nhau (trên/dưới/trái/phải) nối bằng một **cạnh** nếu **không** phải tường. Ta dùng mảng hướng `DIRS = [(1,0),(-1,0),(0,1),(0,-1)]` và kiểm tra biên `0 <= nr < m` và `0 <= nc < n` trước khi đọc ô.
#### Code demo

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
grid = examples[0][0]
m, n = len(grid), len(grid[0])
r, c = 1, 2
neighbors = []
for dr, dc in DIRS:
    nr, nc = r + dr, c + dc
    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != -1:
        neighbors.append((nr, nc))
print("ô", (r, c), "có hàng xóm đi được:", neighbors)

ô (1, 2) có hàng xóm đi được: [(2, 2), (0, 2), (1, 1)]


### 3.2. BFS và đường đi ngắn nhất không trọng số

**BFS** (duyệt theo chiều rộng) dùng **hàng đợi** (`collections.deque`): lấy một nút ra (`popleft`), đưa các hàng xóm chưa thăm vào (`append`). BFS thăm các nút **theo thứ tự khoảng cách tăng dần**: nút ở khoảng cách `d` luôn được thăm trước nút ở khoảng cách `d + 1`. Vì vậy **lần đầu tiên** BFS chạm một nút, khoảng cách ghi nhận là **ngắn nhất**.
#### Code demo

In [3]:
q = deque([(0, 0)])
dist = {(0, 0): 0}
small = [[0, 0, 0], [0, 0, 0]]
while q:
    r, c = q.popleft()
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 2 and 0 <= nc < 3 and (nr, nc) not in dist:
            dist[(nr, nc)] = dist[(r, c)] + 1
            q.append((nr, nc))
print(dist)

{(0, 0): 0, (1, 0): 1, (0, 1): 1, (1, 1): 2, (0, 2): 2, (1, 2): 3}


### 3.3. BFS đa nguồn (multi-source BFS)

Thay vì bắt đầu từ **một** nút, ta đưa **nhiều nút nguồn** (tất cả cổng, khoảng cách `0`) vào hàng đợi **cùng lúc**. BFS lan ra đồng thời từ mọi nguồn như nhiều giọt mực loang, và mỗi ô nhận khoảng cách tới **nguồn gần nhất**. Ta chỉ cần **một** lần BFS cho cả bài thay vì `mn` lần.
#### Code demo

In [4]:
sources = deque([(0, 0), (0, 4)])
width = 5
dist = {s: 0 for s in sources}
while sources:
    r, c = sources.popleft()
    for dc in (-1, 1):
        nc = c + dc
        if 0 <= nc < width and (r, nc) not in dist:
            dist[(r, nc)] = dist[(r, c)] + 1
            sources.append((r, nc))
print([dist[(0, c)] for c in range(width)])           # [0, 1, 2, 1, 0]: mỗi ô nhận khoảng cách tới nguồn gần nhất

[0, 1, 2, 1, 0]


### 3.4. Sửa tại chỗ (in-place) và dùng chính lưới làm bảng khoảng cách

Đề yêu cầu **sửa `rooms`**: ta ghi khoảng cách **trực tiếp** vào ô. Giá trị `INF` còn đóng vai **"chưa thăm"**: ô có `INF` là phòng chưa được gán khoảng cách, nên **không cần** mảng `visited` riêng. Lưu ý: `rooms = [...]` (gán lại tên biến cục bộ) **không** thay đổi lưới của người gọi; phải sửa từng ô `rooms[r][c] = ...`.
#### Code demo

In [5]:
def wrong(g):
    g = [[7]]                       # chỉ gán lại tên biến cục bộ


def right(g):
    g[0][0] = 7                     # sửa ô của lưới gốc


a = [[1]]
wrong(a)
print(a)                            # [[1]]
right(a)
print(a)                            # [[7]]

[[1]]
[[7]]


### 3.5. Vì sao BFS từ cổng thay vì từ phòng

Có thể có `mn` phòng trống nhưng thường ít cổng; hơn nữa **một** BFS đa nguồn từ cổng đã tính khoảng cách của **mọi** phòng. BFS từ từng phòng phải làm lại gần như toàn bộ công việc cho mỗi phòng: `O((mn)²)` — với `250 × 250` là `~4·10^9`, **không khả thi**.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với ví dụ 1, cổng ở `(0,2)` và `(3,0)`. Hình dung "sóng lan" từ hai cổng, mỗi bước lan một ô sang bốn hướng, không qua tường `-1`:

- Bước 1: từ `(0,2)` lan tới `(0,3)` và `(1,2)` (khoảng cách `1`); từ `(3,0)` lan tới `(2,0)` (`1`).
- Bước 2: từ `(1,2)` tới `(1,1)` và `(2,2)` (`2`); từ `(2,0)` tới `(1,0)` (`2`).
- Bước 3: từ `(1,1)` đã có `2` nhưng ô `(0,0)` được tới từ `(1,0)` (`3`); `(3,2)` được tới từ `(2,2)` (`3`).
- Bước 4: `(3,3)` được tới từ `(3,2)` (`4`).

Kết quả khớp đầu ra mẫu. **Cách nghĩ:** khoảng cách tới cổng gần nhất = "thời điểm sóng từ **mọi** cổng đầu tiên chạm ô đó".

## 5. Cách 1 — Brute Force / baseline: BFS từ từng phòng trống

**Ý tưởng.** Với **mỗi** phòng trống, chạy một BFS để tìm khoảng cách tới cổng gần nhất (dừng khi chạm cổng đầu tiên), ghi lại vào ô; BFS dùng bản sao khoảng cách để không làm hỏng lưới gốc trong lúc tính.
#### Code demo

In [6]:
def walls_and_gates_per_room(rooms):
    m, n = len(rooms), len(rooms[0])
    original = copy.deepcopy(rooms)                    # đọc từ bản gốc, ghi vào rooms
    for sr in range(m):
        for sc in range(n):
            if original[sr][sc] != INF:
                continue
            seen = {(sr, sc)}
            queue = deque([(sr, sc, 0)])
            best = INF
            while queue:
                r, c, d = queue.popleft()
                if original[r][c] == 0:
                    best = d                            # BFS: lần đầu chạm cổng là gần nhất
                    break
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and original[nr][nc] != -1 and (nr, nc) not in seen:
                        seen.add((nr, nc))
                        queue.append((nr, nc, d + 1))
            rooms[sr][sc] = best


for start, expected in examples:
    grid = copy.deepcopy(start)
    walls_and_gates_per_room(grid)
    print("khớp kỳ vọng:", grid == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp (Cách 1).** Gọi `N = m·n`. Mỗi BFS tối đa `O(N)`, có tới `N` phòng ⇒ **Time `O(N²)`**; với `N = 62 500` là `~4·10^9` — **quá chậm**. **Space `O(N)`**.

**Nút thắt.** Các BFS lặp lại việc lan truyền trên cùng một lưới. Đảo ngược vai trò: lan **từ cổng** ra mọi phòng chỉ một lần.

## 6. Ý tưởng thuật toán / bước đột phá

**BFS đa nguồn từ các cổng:**

1. Đưa **mọi cổng** (`rooms[r][c] == 0`) vào hàng đợi (khoảng cách `0`).
2. Lấy `(r, c)` ra; với mỗi hàng xóm `(nr, nc)` **trong biên** mà **là `INF`** (phòng trống chưa thăm): gán `rooms[nr][nc] = rooms[r][c] + 1` và đưa vào hàng đợi.
3. Tường `-1` bị bỏ qua; cổng `0` không phải `INF` nên không bị ghi đè; phòng đã có khoảng cách (≠ `INF`) không bị thăm lại.

BFS thăm theo khoảng cách tăng dần, nên giá trị gán lần đầu là **ngắn nhất**. Mỗi ô vào hàng đợi **tối đa một lần** ⇒ `O(mn)`.

## 7. Các cách tiếp theo

### Cách 2 — BFS đa nguồn từ các cổng (cách nộp chuẩn)

**Giải thích.** Như mục 6; dùng chính `rooms` làm bảng khoảng cách (`INF` = chưa thăm).
#### Code demo

In [7]:
def walls_and_gates_multi_bfs(rooms):
    m, n = len(rooms), len(rooms[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if rooms[r][c] == 0)    # mọi cổng là nguồn
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] == INF:              # phòng trống chưa thăm
                rooms[nr][nc] = rooms[r][c] + 1
                queue.append((nr, nc))


for start, expected in examples:
    grid = copy.deepcopy(start)
    walls_and_gates_multi_bfs(grid)
    print("khớp kỳ vọng:", grid == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


In [8]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def wallsAndGates(self, rooms: List[List[int]]) -> None:
        """
        Do not return anything, modify rooms in-place instead.
        """
        m, n = len(rooms), len(rooms[0])
        inf = 2 ** 31 - 1
        q = deque([(i, j) for i in range(m) for j in range(n) if rooms[i][j] == 0])
        d = 0
        while q:
            d += 1
            for _ in range(len(q)):
                i, j = q.popleft()
                for a, b in [[0, 1], [0, -1], [1, 0], [-1, 0]]:
                    x, y = i + a, j + b
                    if 0 <= x < m and 0 <= y < n and rooms[x][y] == inf:
                        rooms[x][y] = d
                        q.append((x, y))


g = copy.deepcopy(examples[0][0])
Solution().wallsAndGates(g)
print(g == examples[0][1])

True


**Độ phức tạp Cách 2:** **Time `O(mn)`** (mỗi ô vào hàng đợi tối đa một lần, mỗi ô xét 4 hàng xóm), **Space `O(mn)`** (hàng đợi, trường hợp xấu toàn cổng).

### Cách 3 — DFS từ từng cổng, có cắt tỉa

**Giải thích.** Từ mỗi cổng chạy DFS: đến ô `(r, c)` với khoảng cách hiện tại `d`, nếu `rooms[r][c] <= d` (ô đã có khoảng cách tốt hơn hoặc bằng, hoặc là tường `-1`) thì **dừng** (cắt tỉa); ngược lại cập nhật `rooms[r][c] = d` và đi tiếp `d + 1` sang 4 hướng. Cắt tỉa này là bắt buộc để không lặp vô hạn.
#### Code demo

In [9]:
def walls_and_gates_dfs(rooms):
    m, n = len(rooms), len(rooms[0])
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 100_000))

    def dfs(r, c, d):
        if r < 0 or r >= m or c < 0 or c >= n or rooms[r][c] < d:      # tường (-1 < d) hoặc đã có đường ngắn hơn
            return
        rooms[r][c] = d
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] > d + 1:     # chỉ đi tiếp nếu cải thiện được
                dfs(nr, nc, d + 1)

    for r in range(m):
        for c in range(n):
            if rooms[r][c] == 0:
                dfs(r, c, 0)


for start, expected in examples:
    grid = copy.deepcopy(start)
    walls_and_gates_dfs(grid)
    print("khớp kỳ vọng:", grid == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp Cách 3:** mỗi ô có thể bị cập nhật nhiều lần (mỗi lần tìm được đường ngắn hơn): worst case **Time `O(mn · số_cổng)`** (thường tốt hơn nhiều); **Space `O(mn)`** cho ngăn xếp đệ quy (cần nâng giới hạn đệ quy). Đúng nhưng **kém BFS** về độ phức tạp: ở phần test, trên lưới `60 × 60` DFS đã chậm hơn BFS khoảng hai bậc độ lớn, còn trên lưới `250 × 250` nó mất hàng phút (nhiều ô bị cập nhật đi cập nhật lại).

### Cách 4 — BFS đa nguồn theo từng lớp (đếm khoảng cách bằng vòng lặp ngoài)

**Giải thích.** Thay vì tính `rooms[r][c] + 1`, ta duyệt **từng lớp** của hàng đợi và tăng biến `dist`. Cách này giống Cách 2 nhưng làm rõ khái niệm "lớp khoảng cách" và dễ mở rộng cho bài toán "đếm số lớp".
#### Code demo

In [10]:
def walls_and_gates_layered(rooms):
    m, n = len(rooms), len(rooms[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if rooms[r][c] == 0)
    dist = 0
    while queue:
        dist += 1
        for _ in range(len(queue)):                    # xử lý đúng một lớp
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] == INF:
                    rooms[nr][nc] = dist
                    queue.append((nr, nc))


for start, expected in examples:
    grid = copy.deepcopy(start)
    walls_and_gates_layered(grid)
    print("khớp kỳ vọng:", grid == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp Cách 4:** **Time `O(mn)`**, **Space `O(mn)`**.

### Cách 5 — Lặp nới lỏng (Bellman–Ford): quét lưới tới khi không còn thay đổi

**Giải thích.** Lặp: với mỗi phòng, thử `rooms[r][c] = min(rooms[r][c], 1 + min(hàng xóm không phải tường))`. Lặp lại đến khi một lượt quét **không đổi gì**. Mỗi lượt lan thêm ít nhất một "lớp" nên số lượt `<=` khoảng cách lớn nhất `+ 1`.
#### Code demo

In [11]:
def walls_and_gates_relax(rooms):
    m, n = len(rooms), len(rooms[0])
    changed = True
    while changed:
        changed = False
        for r in range(m):
            for c in range(n):
                if rooms[r][c] in (-1, 0):
                    continue
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] >= 0 and rooms[nr][nc] != INF and rooms[nr][nc] + 1 < rooms[r][c]:
                        rooms[r][c] = rooms[nr][nc] + 1
                        changed = True


for start, expected in examples:
    grid = copy.deepcopy(start)
    walls_and_gates_relax(grid)
    print("khớp kỳ vọng:", grid == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp Cách 5:** **Time `O(mn · D)`** với `D` là khoảng cách lớn nhất (có thể `~mn/2` ở mê cung ngoằn ngoèo ⇒ `O((mn)²)` worst); **Space `O(1)`** phụ. Hữu ích khi muốn tránh hàng đợi, nhưng chậm hơn BFS.

> Cách 2 (hoặc Cách 4) là bản nộp chuẩn. Cách 3 cho thấy DFS cần cắt tỉa. Cách 5 dạy tư duy "nới lỏng cạnh" của đường đi ngắn nhất.

## 8. TRACE từng bước

Ta in lưới sau mỗi lớp của BFS đa nguồn cho ví dụ 1 (Cách 4), kèm các ô mới được gán trong lớp đó.
#### Code demo

In [12]:
def trace_layers(rooms):
    m, n = len(rooms), len(rooms[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if rooms[r][c] == 0)
    print("nguồn (cổng):", list(queue))
    dist = 0
    while queue:
        dist += 1
        assigned = []
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] == INF:
                    rooms[nr][nc] = dist
                    queue.append((nr, nc))
                    assigned.append((nr, nc))
        print(f"\nlớp {dist}: gán khoảng cách {dist} cho {assigned}")
        print(show(rooms))


trace_layers(copy.deepcopy(examples[0][0]))

nguồn (cổng): [(0, 2), (3, 0)]

lớp 1: gán khoảng cách 1 cho [(1, 2), (0, 3), (2, 0)]
  INF  -1   0   1
  INF INF   1  -1
    1  -1 INF  -1
    0  -1 INF INF

lớp 2: gán khoảng cách 2 cho [(2, 2), (1, 1), (1, 0)]
  INF  -1   0   1
    2   2   1  -1
    1  -1   2  -1
    0  -1 INF INF

lớp 3: gán khoảng cách 3 cho [(3, 2), (0, 0)]
    3  -1   0   1
    2   2   1  -1
    1  -1   2  -1
    0  -1   3 INF

lớp 4: gán khoảng cách 4 cho [(3, 3)]
    3  -1   0   1
    2   2   1  -1
    1  -1   2  -1
    0  -1   3   4

lớp 5: gán khoảng cách 5 cho []
    3  -1   0   1
    2   2   1  -1
    1  -1   2  -1
    0  -1   3   4


Bảng tóm tắt (ví dụ 1):

| lớp (khoảng cách) | các ô được gán |
|---|---|
| 1 | `(0,3)`, `(1,2)`, `(2,0)` |
| 2 | `(1,1)`, `(2,2)`, `(1,0)` |
| 3 | `(0,0)`, `(3,2)` |
| 4 | `(3,3)` |

Mỗi ô nhận giá trị **ngay lần đầu** được hàng đợi chạm tới — đó là khoảng cách ngắn nhất.

## 9. Code nộp LeetCode

Bản nộp dùng BFS đa nguồn theo lớp (Cách 2/4). Đúng `class Solution`, đúng `wallsAndGates(self, rooms: List[List[int]]) -> None`, sửa `rooms` **tại chỗ**.

In [13]:
class Solution:
    def wallsAndGates(self, rooms: List[List[int]]) -> None:
        """
        Do not return anything, modify rooms in-place instead.
        """
        m, n = len(rooms), len(rooms[0])
        inf = 2 ** 31 - 1
        q = deque([(i, j) for i in range(m) for j in range(n) if rooms[i][j] == 0])   # mọi cổng là nguồn
        d = 0
        while q:
            d += 1
            for _ in range(len(q)):                                                   # một lớp khoảng cách
                i, j = q.popleft()
                for a, b in [[0, 1], [0, -1], [1, 0], [-1, 0]]:
                    x, y = i + a, j + b
                    if 0 <= x < m and 0 <= y < n and rooms[x][y] == inf:              # phòng trống chưa thăm
                        rooms[x][y] = d
                        q.append((x, y))


sol = Solution()
for start, expected in examples:
    grid = copy.deepcopy(start)
    sol.wallsAndGates(grid)
    assert grid == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **BFS cho khoảng cách ngắn nhất:** trong đồ thị không trọng số, BFS từ một tập nguồn thăm các nút theo khoảng cách **không giảm** tới tập nguồn; nút đầu tiên chạm được ở lớp `d` có khoảng cách ngắn nhất đúng bằng `d`. Thêm một "siêu nguồn" ảo nối tới mọi cổng bằng cạnh độ dài `0` đưa bài về BFS một nguồn, khoảng cách tới cổng gần nhất = khoảng cách tới siêu nguồn.
- **Mỗi ô vào hàng đợi một lần:** ô chỉ được gán khi còn `INF`; sau khi gán nó khác `INF`, nên không bị gán lại.
- **Tường và cổng không bị ghi đè:** điều kiện `== INF` loại `-1` và `0`.
- **Phòng không tới được:** không bao giờ bị chạm ⇒ giữ `INF`, đúng yêu cầu.
- **Đúng 4 hướng:** `DIRS` chỉ gồm bước ngang/dọc.
- **Cách 3:** DFS với cắt tỉa `rooms[r][c] < d ⇒ dừng` và chỉ đi tiếp khi cải thiện được ⇒ khi dừng, mọi ô đạt giá trị nhỏ nhất qua mọi cổng (chứng minh bằng quy nạp theo khoảng cách). **Cách 5:** nới lỏng cạnh cho tới điểm cố định = đường đi ngắn nhất.

## 11. Phân tích Time / Space Complexity

Gọi `N = m·n` (tối đa `62 500`), `G` số cổng, `D` khoảng cách lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. BFS từ từng phòng | `O(N²)` | `O(N)` |
| 2/4. BFS đa nguồn từ cổng | `O(N)` | `O(N)` |
| 3. DFS từ từng cổng, cắt tỉa | `O(N·G)` worst | `O(N)` (ngăn xếp) |
| 5. Nới lỏng lặp | `O(N·D)` | `O(1)` phụ |

Với `N = 62 500`: BFS đa nguồn cỡ `2.5·10^5` thao tác (vài chục ms); Cách 1 cỡ `4·10^9` (không chạy nổi). Không có khác biệt average/worst ở Cách 2/4: mỗi ô vào hàng đợi tối đa một lần.

## 12. Bộ test

`oracle` là brute force độc lập: dựng **ma trận khoảng cách mọi cặp ô** bằng **Floyd–Warshall** trên đồ thị lưới (chỉ cho lưới nhỏ, tối đa ~25 ô), rồi khoảng cách của mỗi phòng là `min` khoảng cách tới các cổng (hoặc `INF` nếu không tới được).
#### Code demo

In [14]:
def oracle(rooms):
    m, n = len(rooms), len(rooms[0])
    cells = [(r, c) for r in range(m) for c in range(n)]
    index = {cell: i for i, cell in enumerate(cells)}
    big = 10 ** 9
    k = len(cells)
    dist = [[big] * k for _ in range(k)]
    for (r, c) in cells:
        if rooms[r][c] == -1:
            continue
        i = index[(r, c)]
        dist[i][i] = 0
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] != -1:
                dist[i][index[(nr, nc)]] = 1
    for mid in range(k):
        for i in range(k):
            if dist[i][mid] == big:
                continue
            for j in range(k):
                if dist[i][mid] + dist[mid][j] < dist[i][j]:
                    dist[i][j] = dist[i][mid] + dist[mid][j]
    gates = [index[(r, c)] for (r, c) in cells if rooms[r][c] == 0]
    out = [row[:] for row in rooms]
    for (r, c) in cells:
        if rooms[r][c] == INF:
            best = min((dist[index[(r, c)]][g] for g in gates), default=big)
            out[r][c] = best if best < big else INF
    return out


def run_inplace(fn, grid):
    g = copy.deepcopy(grid)
    fn(g)
    return g


impls = {
    "per_room": walls_and_gates_per_room,
    "multi_bfs": walls_and_gates_multi_bfs,
    "dfs": walls_and_gates_dfs,
    "layered": walls_and_gates_layered,
    "relax": walls_and_gates_relax,
    "submit": sol.wallsAndGates,
}

for start, expected in examples:
    assert oracle(start) == expected
    for key, fn in impls.items():
        assert run_inplace(fn, start) == expected, (key, start)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([[-1]], "một ô tường"),
    ([[0]], "một cổng"),
    ([[INF]], "một phòng trống không có cổng: giữ INF"),
    ([[0, INF, INF, INF]], "một hàng, cổng ở đầu"),
    ([[INF, INF, 0]], "cổng ở cuối hàng"),
    ([[0], [INF], [INF]], "một cột"),
    ([[0, -1, INF]], "tường chặn: phòng bên phải không tới được"),
    ([[INF, INF], [INF, INF]], "không có cổng"),
    ([[0, 0], [0, 0]], "toàn cổng"),
    ([[-1, -1], [-1, -1]], "toàn tường"),
    ([[0, INF, -1, INF, 0]], "hai cổng chặn nhau bởi tường"),
    ([[INF, INF, INF], [INF, 0, INF], [INF, INF, INF]], "cổng ở giữa"),
    ([[0, INF, INF], [-1, -1, INF], [INF, INF, INF]], "đường vòng"),
]
for start, label in edge_cases:
    truth = oracle(start)
    for key, fn in impls.items():
        assert run_inplace(fn, start) == truth, (key, label, run_inplace(fn, start), truth)
    print("PASS", label)

PASS một ô tường
PASS một cổng
PASS một phòng trống không có cổng: giữ INF
PASS một hàng, cổng ở đầu
PASS cổng ở cuối hàng
PASS một cột
PASS tường chặn: phòng bên phải không tới được
PASS không có cổng
PASS toàn cổng
PASS toàn tường
PASS hai cổng chặn nhau bởi tường
PASS cổng ở giữa
PASS đường vòng


### 12.2. Tính chất và phản ví dụ

- Tường (`-1`) và cổng (`0`) **không đổi**; mọi phòng trống thành số `>= 1` hoặc `INF`.
- Hai ô **kề nhau** (không tường) có khoảng cách **chênh nhau tối đa `1`** (tính 1-Lipschitz).
- Mọi phòng có giá trị `d >= 1` có **một hàng xóm** với giá trị `d - 1` (đường ngắn nhất).
- **Phản ví dụ** cho lỗi BFS từ từng phòng khi không dùng bản sao hoặc lỗi không cắt tỉa: với bản **gán lại tham số** (không sửa tại chỗ), lưới gốc không đổi.
#### Code demo

In [16]:
rng = random.Random(286)


def random_grid(m, n, rng, p_wall=0.25, p_gate=0.15):
    grid = []
    for _ in range(m):
        row = []
        for _ in range(n):
            x = rng.random()
            row.append(-1 if x < p_wall else (0 if x < p_wall + p_gate else INF))
        grid.append(row)
    return grid


for _ in range(300):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    start = random_grid(m, n, rng)
    out = run_inplace(sol.wallsAndGates, start)
    for r in range(m):
        for c in range(n):
            if start[r][c] in (-1, 0):
                assert out[r][c] == start[r][c]
            else:
                assert out[r][c] == INF or out[r][c] >= 1
                if out[r][c] != INF:
                    d = out[r][c]
                    nbrs = [out[r + dr][c + dc] for dr, dc in DIRS if 0 <= r + dr < m and 0 <= c + dc < n]
                    assert (d - 1) in nbrs                                    # có hàng xóm ở khoảng cách d - 1
                    for v in nbrs:
                        if v not in (-1, INF):
                            assert abs(v - d) <= 1                            # chênh tối đa 1
print("PASS tính chất (tường/cổng giữ nguyên, hàng xóm d-1, Lipschitz-1) trên 300 lưới ngẫu nhiên")


def sai_gan_lai_tham_so(rooms):
    rooms = [[0 for _ in row] for row in rooms]                     # SAI: gán lại tên biến, không sửa lưới của người gọi
    return rooms


g = copy.deepcopy(examples[0][0])
sai_gan_lai_tham_so(g)
print("sau khi gọi bản sai, lưới gốc vẫn:", g == examples[0][0])
assert g == examples[0][0]

PASS tính chất (tường/cổng giữ nguyên, hàng xóm d-1, Lipschitz-1) trên 300 lưới ngẫu nhiên
sau khi gọi bản sai, lưới gốc vẫn: True


### 12.3. Đối chiếu ngẫu nhiên (có seed) và lưới lớn
#### Code demo

In [17]:
rng = random.Random(4286)
for trial in range(1500):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    start = random_grid(m, n, rng, p_wall=rng.choice([0.1, 0.3, 0.5]), p_gate=rng.choice([0.05, 0.2]))
    truth = oracle(start)
    for key, fn in impls.items():
        assert run_inplace(fn, start) == truth, (key, start)
print("PASS 1500 lưới nhỏ ngẫu nhiên (nhiều tường/cổng): mọi cách khớp oracle Floyd–Warshall")

# Lưới lớn 250 x 250: so BFS đa nguồn, theo lớp và bản nộp (DFS cắt tỉa quá chậm ở cỡ này, xem lưới 60 x 60 bên dưới)
big = random_grid(250, 250, rng, p_wall=0.2, p_gate=0.002)
ref = None
for key in ("multi_bfs", "layered", "submit"):
    t0 = time.perf_counter()
    g = copy.deepcopy(big)
    impls[key](g)
    dt = (time.perf_counter() - t0) * 1000
    ref = g if ref is None else ref
    assert g == ref, key
    print(f"{key:<10} {dt:9.1f} ms")
# Lưới vừa 60 x 60: thêm nới lỏng lặp và BFS từng phòng (chậm hơn nhiều trên lưới lớn)
mid = random_grid(60, 60, rng, p_wall=0.2, p_gate=0.01)
ref_mid = run_inplace(walls_and_gates_multi_bfs, mid)
for key in ("per_room", "relax", "dfs", "layered", "submit"):
    t0 = time.perf_counter()
    got = run_inplace(impls[key], mid)
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref_mid, key
    print(f"60x60 {key:<10} {dt:9.1f} ms")
print("PASS lưới lớn: các cách khớp nhau")

PASS 1500 lưới nhỏ ngẫu nhiên (nhiều tường/cổng): mọi cách khớp oracle Floyd–Warshall
multi_bfs       36.2 ms
layered         35.9 ms
submit          40.3 ms


60x60 per_room       182.3 ms
60x60 relax           38.1 ms


60x60 dfs            547.7 ms
60x60 layered          2.1 ms
60x60 submit           2.2 ms
PASS lưới lớn: các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: BFS từ từng phòng thay vì từ các cổng
Cách 1 đúng nhưng chậm `O(N²)`. **Cách sửa:** đảo ngược — BFS đa nguồn từ mọi cổng.

### Lỗi 2: Không đánh dấu ô đã thăm (hoặc không kiểm tra `== INF`)
#### Code sai

In [18]:
def sai_khong_danh_dau(rooms, limit=200_000):
    m, n = len(rooms), len(rooms[0])
    queue = deque((r, c, 0) for r in range(m) for c in range(n) if rooms[r][c] == 0)
    steps = 0
    while queue and steps < limit:                    # limit chỉ để demo, tránh treo notebook
        steps += 1
        r, c, d = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] != -1:      # SAI: cho cả ô đã thăm và cả cổng
                rooms[nr][nc] = min(rooms[nr][nc], d + 1) if rooms[nr][nc] != 0 else 0
                queue.append((nr, nc, d + 1))
    return steps


g = copy.deepcopy(examples[0][0])
print("bản sai chạy", sai_khong_danh_dau(g), "bước (đạt giới hạn demo) trong khi BFS đúng chỉ cần khoảng", sum(1 for row in g for v in row if v != -1), "ô")

bản sai chạy 200000 bước (đạt giới hạn demo) trong khi BFS đúng chỉ cần khoảng 11 ô


**Vì sao sai:** không loại ô đã thăm nên hàng đợi **không bao giờ cạn** (đi qua lại giữa hai ô). **Code đúng:** chỉ xử lý ô có `rooms[nr][nc] == INF` rồi gán ngay (ô thôi là `INF`).

### Lỗi 3: DFS không cắt tỉa
#### Code sai

In [19]:
def sai_dfs_khong_cat_tia(rooms):
    m, n = len(rooms), len(rooms[0])
    calls = 0

    def dfs(r, c, d, depth):
        nonlocal calls
        calls += 1
        if calls > 20000 or depth > 400:                           # chặn demo
            return
        if r < 0 or r >= m or c < 0 or c >= n or rooms[r][c] == -1:
            return
        rooms[r][c] = min(rooms[r][c], d)                          # SAI: không dừng khi ô đã có đường ngắn hơn
        for dr, dc in DIRS:
            dfs(r + dr, c + dc, d + 1, depth + 1)

    for r in range(m):
        for c in range(n):
            if rooms[r][c] == 0:
                dfs(r, c, 0, 0)
    return calls


g = copy.deepcopy(examples[0][0])
print("số lời gọi DFS sai (bị chặn ở 20000):", sai_dfs_khong_cat_tia(g))
print("bản đúng (DFS cắt tỉa) chỉ cần thao tác trên", sum(1 for row in examples[0][0] for v in row if v != -1), "ô, mỗi ô vài lần")

số lời gọi DFS sai (bị chặn ở 20000): 21006
bản đúng (DFS cắt tỉa) chỉ cần thao tác trên 11 ô, mỗi ô vài lần


**Vì sao sai:** DFS đi lại qua cùng ô vô hạn lần (không có điều kiện dừng khi không cải thiện). **Code đúng:** dừng khi `rooms[r][c] < d` (hoặc chỉ đi tiếp nếu `rooms[nr][nc] > d + 1`).

### Lỗi 4: Cho phép đi chéo hoặc vượt tường
#### Code sai

In [20]:
def sai_di_cheo(rooms):
    m, n = len(rooms), len(rooms[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if rooms[r][c] == 0)
    dirs8 = DIRS + [(1, 1), (1, -1), (-1, 1), (-1, -1)]            # SAI: đề chỉ cho 4 hướng
    while queue:
        r, c = queue.popleft()
        for dr, dc in dirs8:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1
                queue.append((nr, nc))


g = run_inplace(sai_di_cheo, examples[0][0])
print("đúng:", run_inplace(sol.wallsAndGates, examples[0][0])[0])
print("sai  :", g[0])

đúng: [3, -1, 0, 1]
sai  : [2, -1, 0, 1]


**Vì sao sai:** đi chéo rút ngắn khoảng cách sai (và có thể "lách" giữa hai tường). Chỉ dùng `DIRS` 4 hướng.

### Lỗi 5: Gán lại tham số thay vì sửa tại chỗ
Xem `sai_gan_lai_tham_so` ở phần test: sau khi gọi, lưới gốc không đổi. **Code đúng:** sửa từng ô `rooms[r][c] = ...`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**khoảng cách ngắn nhất tới điểm/nguồn gần nhất**" trên lưới → **BFS đa nguồn**: đưa **mọi nguồn** vào hàng đợi lúc đầu.
- "Đảo ngược vai trò": thay vì từ **mỗi đích** tìm nguồn, hãy **lan từ nguồn** (hoặc từ tập nguồn) một lần.
- Dùng chính lưới làm bảng khoảng cách, giá trị `INF` làm trạng thái "chưa thăm" (không cần mảng `visited`).
- BFS theo lớp: `for _ in range(len(queue))` + biến `dist`.
- Luôn kiểm tra **biên** trước khi đọc ô; chỉ **4 hướng**.
- Họ hàng: **01 Matrix** (542), **Rotting Oranges** (994), **As Far from Land as Possible** (1162), **Shortest Distance from All Buildings** (317), **Map of Highest Peak** (1765).
- Thử: không có cổng, một cổng bị tường bao, toàn tường, một hàng/một cột.

## 15. Biến thể của bài

1. **01 Matrix** (LeetCode 542): khoảng cách tới số `0` gần nhất — cùng thuật toán.
2. **Rotting Oranges** (994): thời gian để thối hết — BFS đa nguồn theo lớp.
3. **As Far from Land as Possible** (1162): ô nước xa đất nhất.
4. **Shortest Distance from All Buildings** (317): BFS từ từng tòa nhà, cộng dồn khoảng cách.
5. **Thêm trọng số** (ô khó đi tốn nhiều hơn): Dijkstra thay BFS.
6. Trả về **cổng nào gần nhất** (lưu chỉ số cổng cùng khoảng cách).
#### Code demo

In [21]:
def nearest_gate_id(rooms):
    """Với mỗi phòng trống, trả (khoảng cách, chỉ số cổng gần nhất) bằng BFS đa nguồn (hòa: cổng nào tới trước)."""
    m, n = len(rooms), len(rooms[0])
    gates = [(r, c) for r in range(m) for c in range(n) if rooms[r][c] == 0]
    result = [[None] * n for _ in range(m)]
    queue = deque()
    for gid, (r, c) in enumerate(gates):
        result[r][c] = (0, gid)
        queue.append((r, c))
    while queue:
        r, c = queue.popleft()
        d, gid = result[r][c]
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] == INF and result[nr][nc] is None:
                result[nr][nc] = (d + 1, gid)
                queue.append((nr, nc))
    return result


res = nearest_gate_id(examples[0][0])
print(res[0])          # hàng đầu: (khoảng cách, id cổng) hoặc None cho tường

[(3, 1), None, (0, 0), (1, 0)]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS từ từng phòng | mỗi phòng một BFS tìm cổng gần nhất | `O(N²)` | `O(N)` | baseline, hiểu bài |
| 2/4. BFS đa nguồn | lan đồng thời từ mọi cổng | `O(N)` | `O(N)` | **cách nộp chuẩn** |
| 3. DFS cắt tỉa | DFS từ từng cổng, dừng khi không cải thiện | `O(N·G)` worst | `O(N)` | khi thích DFS; cần nâng giới hạn đệ quy |
| 5. Nới lỏng lặp | cập nhật `min(1 + hàng xóm)` tới điểm cố định | `O(N·D)` | `O(1)` | không cần hàng đợi |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán lưới sau khi gọi `wallsAndGates` cho `[[0, INF, INF]]` và cho `[[0, -1, INF]]`.

**Bài 2 (tính bằng tay).** Điền khoảng cách cho lưới `[[INF, INF, INF], [INF, -1, INF], [0, INF, INF]]` (cổng ở góc dưới trái, tường ở giữa), ghi rõ từng lớp BFS.

**Bài 3 (điền TODO).** Hoàn thiện `gates_todo(rooms)` theo BFS đa nguồn.
#### Code demo

In [22]:
def gates_todo(rooms):
    m, n = len(rooms), len(rooms[0])
    # TODO 1: queue = deque chứa mọi (r, c) có rooms[r][c] == 0
    # TODO 2: while queue: lấy (r, c) ra
    # TODO 3: với mỗi hướng (dr, dc) trong DIRS: tính (nr, nc), kiểm tra biên
    # TODO 4: nếu rooms[nr][nc] == INF: gán rooms[r][c] + 1 và đưa (nr, nc) vào queue
    pass


# Tự làm xong rồi mới mở test:
# for start, expected in examples:
#     g = copy.deepcopy(start); gates_todo(g); assert g == expected

**Bài 4 (tự viết lại).** Viết `farthest_room(rooms)` trả khoảng cách **lớn nhất** (trong các phòng tới được) sau khi đã điền khoảng cách, và `unreachable_count(rooms)` đếm số phòng không tới được.

**Bài 5 (biến thể / mini challenge).** Viết `zero_one_matrix(mat)` (LeetCode 542): với ma trận `0/1`, khoảng cách tới số `0` gần nhất của mỗi ô (4 hướng), bằng BFS đa nguồn; đối chiếu với brute force `O((mn)²)` trên lưới nhỏ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: lan từ cổng sang phải từng bước; tường chặn. Bài 2: bắt đầu từ cổng `(2,0)`. Bài 5: coi các ô `0` là cổng, các ô `1` là phòng.
- **Mức 2 (vừa):** Bài 1: `[[0, 1, 2]]` và `[[0, -1, INF]]`. Bài 2: lớp 1: `(1,0)` và `(2,1)`; lớp 2: `(0,0)` và `(2,2)`; lớp 3: `(0,1)` và `(1,2)`; lớp 4: `(0,2)`. Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 2 kết quả: `[[2, 3, 4], [1, -1, 3], [0, 1, 2]]`. Bài 4: duyệt lưới lấy `max` của các giá trị `!= INF` và `!= -1`; đếm ô `== INF` ban đầu còn lại. Bài 5: nguồn là ô `0`, ô `1` ban đầu đặt `INF`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
# Bài 1
assert run_inplace(sol.wallsAndGates, [[0, INF, INF]]) == [[0, 1, 2]]
assert run_inplace(sol.wallsAndGates, [[0, -1, INF]]) == [[0, -1, INF]]

# Bài 2
b2 = [[INF, INF, INF], [INF, -1, INF], [0, INF, INF]]
assert run_inplace(sol.wallsAndGates, b2) == [[2, 3, 4], [1, -1, 3], [0, 1, 2]] == oracle(b2)


# Bài 3
def gates_todo_ans(rooms):
    m, n = len(rooms), len(rooms[0])
    queue = deque((r, c) for r in range(m) for c in range(n) if rooms[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1
                queue.append((nr, nc))


for start, expected in examples:
    assert run_inplace(gates_todo_ans, start) == expected


# Bài 4
def farthest_room(filled):
    return max((v for row in filled for v in row if v not in (-1, INF)), default=0)


def unreachable_count(filled):
    return sum(1 for row in filled for v in row if v == INF)


f1 = run_inplace(sol.wallsAndGates, examples[0][0])
assert farthest_room(f1) == 4 and unreachable_count(f1) == 0
assert unreachable_count(run_inplace(sol.wallsAndGates, [[0, -1, INF]])) == 1


# Bài 5
def zero_one_matrix(mat):
    m, n = len(mat), len(mat[0])
    dist = [[0 if mat[r][c] == 0 else -1 for c in range(n)] for r in range(m)]
    queue = deque((r, c) for r in range(m) for c in range(n) if mat[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                queue.append((nr, nc))
    return dist


def zero_one_brute(mat):
    m, n = len(mat), len(mat[0])
    zeros = [(r, c) for r in range(m) for c in range(n) if mat[r][c] == 0]
    return [[min(abs(r - zr) + abs(c - zc) for zr, zc in zeros) for c in range(n)] for r in range(m)]


rng = random.Random(17)
for _ in range(200):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[rng.choice([0, 1, 1]) for _ in range(n)] for _ in range(m)]
    if all(v == 1 for row in mat for v in row):
        mat[0][0] = 0
    assert zero_one_matrix(mat) == zero_one_brute(mat)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (BFS đa nguồn theo lớp, sửa tại chỗ)

```python
class Solution:
    def wallsAndGates(self, rooms: List[List[int]]) -> None:
        m, n = len(rooms), len(rooms[0])
        inf = 2 ** 31 - 1
        q = deque([(i, j) for i in range(m) for j in range(n) if rooms[i][j] == 0])
        d = 0
        while q:
            d += 1
            for _ in range(len(q)):
                i, j = q.popleft()
                for a, b in [[0, 1], [0, -1], [1, 0], [-1, 0]]:
                    x, y = i + a, j + b
                    if 0 <= x < m and 0 <= y < n and rooms[x][y] == inf:
                        rooms[x][y] = d
                        q.append((x, y))
```

**Checklist**

- [ ] Đưa **mọi cổng** vào hàng đợi lúc đầu (BFS **đa nguồn**), không BFS từ từng phòng?
- [ ] Chỉ xử lý ô `== INF` (chưa thăm) và gán khoảng cách ngay khi đưa vào hàng đợi?
- [ ] Chỉ **4 hướng**; kiểm tra **biên** trước khi đọc ô; tường `-1` và cổng `0` không bị ghi đè?
- [ ] Sửa **tại chỗ** (không gán lại `rooms`), không trả giá trị?
- [ ] Phòng không tới được giữ `INF`?
- [ ] Đã thử không có cổng, cổng bị tường bao, toàn tường, một hàng/cột, lưới `250 × 250`?
- [ ] Đã đối chiếu với brute force Floyd–Warshall trên lưới nhỏ ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def wallsAndGates(self, rooms: List[List[int]]) -> None`?